# A physically based storm portrait

This notebook renders a thunderstorm from model output the way it would
look to the eye. It covers:

1. the shortest path from a RAMS output file to a rendered image;
2. framing the camera and placing the sun;
3. placing the camera explicitly, in model coordinates;
4. aerial perspective, the haze between the camera and a distant storm;
5. CM1 output with Morrison microphysics.

**How the cloud is made.** skyvista reads each hydrometeor species' mixing
ratio and number concentration. From these it computes the particle size
distribution, then the *extinction*: the fraction of light that the
particles scatter per meter. The extinction of every species goes into a
volume file (OpenVDB). Blender's path tracer, Cycles, then renders that
volume under a physically modelled sun and sky.

**What you need.** Run the notebook in the pixi `blender` environment of
the skyvista repository (`pixi run -e blender jupyter lab`). You also need
Blender 5.0. The paths to Blender and to the model output are in
`notebook_setup.py`, next to this notebook. The model output is read with
`xarrams`, a separate package for RAMS and CM1 output. skyvista itself
works with any xarray `Dataset`.

Each render here uses skyvista's "preview" quality: 16 samples per pixel
and a 960 x 540 image. A render then takes seconds to a few minutes. The
last section says what to change for a final figure.

In [ ]:
import sys
from pathlib import Path

# notebook_setup.py sits next to this notebook. Jupyter starts the kernel
# either in this folder or in the repository root (VS Code's default).
for notebook_folder in (Path.cwd(), Path.cwd() / "examples" / "notebooks"):
    if (notebook_folder / "notebook_setup.py").exists():
        sys.path.insert(0, str(notebook_folder))
        break

import numpy as np
import xarray as xr
import xarrams

import notebook_setup as setup
import skyvista as sv
from skyvista.cloud_optics import cm1_morrison_hydrometeor_species

## 1. Load the model output

`xarrams.read_rams_output` reads RAMS output files into an xarray
`Dataset`. It takes the grid coordinates `x`, `y` and `z` (m) from the
run's `-head.txt` file, and the time from the file name.

The clouds need these variables:

- the mixing ratio of each hydrometeor category (`R?P`, kg/kg);
- its number concentration (`C?P`, #/kg);
- the base-state air density `DN0` (kg/m^3).

`keep_vars` names them. Names absent from the file are skipped: this
run has no drizzle category (`RDP`) and no `DN0` in its lite files.

Two selections follow:

- `isel(time=0)` takes the one output time.
- `sel(z=slice(0, 16500))` drops the RAMS level below the ground, and
  the levels above 16.5 km, which hold no cloud here. Fewer levels make
  the export faster.

`xarrams` reads lazily, so `load()` reads the selected data into memory.

In [ ]:
# RAMS hydrometeor mixing ratios (R?P), number concentrations (C?P) and
# the base-state air density.
RAMS_CLOUD_VARIABLES = [
    "RCP", "RDP", "RRP", "RPP", "RSP", "RAP", "RGP", "RHP",
    "CCP", "CDP", "CRP", "CPP", "CSP", "CAP", "CGP", "CHP",
    "DN0",
]
# Levels kept: from the ground to the highest cloud (m).
LOWEST_HEIGHT, TOP_HEIGHT = 0.0, 16500.0
# RAMS output names count model time from this date.
RUN_START = np.datetime64("2000-01-01T00:00:00")

# One output time of an isolated storm, 94 min into the run.
storm_output = xarrams.read_rams_output(
    [setup.RAMS_STORM_FILE], keep_vars=RAMS_CLOUD_VARIABLES, silent=True
)
storm_clouds = storm_output.isel(time=0).sel(z=slice(LOWEST_HEIGHT, TOP_HEIGHT)).load()

storm_minutes = (storm_clouds["time"].values - RUN_START) / np.timedelta64(1, "m")
print(f"t = {storm_minutes:.1f} min, grid {dict(storm_clouds.sizes)}")
print("variables:", ", ".join(storm_clouds.data_vars))

## 2. The shortest path to an image

Three calls make a render:

1. `sv.Scene()` creates an empty scene.
2. `add_clouds(..., microphysics="rams")` adds the cloud. The
   `microphysics` preset names the variables and the size-distribution
   settings of the RAMS scheme.
3. `to_blender(bundle_dir, render=True)` exports the scene and runs
   Blender. `quiet=True` sends Blender's console output to
   `blender.log` in the bundle, instead of into the notebook.

`BlenderExportConfig.for_clouds()` holds the settings for physical clouds:
the physical sky, a ground plane, many light bounces and automatic
exposure. `quality="preview"` lowers the samples and the image size for
a quick look.

The *bundle* is the directory that `to_blender` writes. It holds the volume
files (`data/`), a manifest (`scene.json`) and the script that builds the
Blender scene. Blender adds a `.blend` file, which opens in the Blender
application, and the image `render/frame_0001.png`.

In [ ]:
quick_bundle_dir = setup.WORK_DIR / "portrait_quick"

# Physical-cloud defaults, at preview quality.
quick_config = sv.BlenderExportConfig.for_clouds(quality="preview")

# Export the cloud and render it with Blender.
sv.Scene().add_clouds(storm_clouds, microphysics="rams").to_blender(
    quick_bundle_dir, config=quick_config, render=True, quiet=True
)
setup.show_image(quick_bundle_dir)

## 3. Frame the camera and place the sun

By default the camera frames the bounding box of everything exported. Four
settings change that framing:

- `camera.azimuth_deg`: the direction from the scene's center to the camera,
  counterclockwise from east (+x). -90 puts the camera south of the storm.
- `camera.elevation_deg`: the camera's angle above the scene's center. A
  negative value puts the camera low, looking up at the storm.
- `camera.distance_factor`: the camera's distance, in bounding-box
  diagonals.
- `camera.lens_mm`: the focal length. 35 mm spans about 54 degrees across.

The sun has an azimuth (the direction *toward* the sun, counterclockwise
from east) and an elevation. The sun here shines from the camera's left
and a little behind the storm, so the towers show light and shadow.

Note that the elevation is measured from the center of the scene, not from
the ground. A large negative elevation on a wide scene can therefore put
the camera underground. skyvista warns when that happens.

In [ ]:
framed_bundle_dir = setup.WORK_DIR / "portrait_framed"
framed_config = sv.BlenderExportConfig.for_clouds(quality="preview")

# A low camera south-east of the storm, looking slightly up.
framed_config.camera.azimuth_deg = -55.0
framed_config.camera.elevation_deg = -8.0
framed_config.camera.distance_factor = 1.1

# The sun 30 degrees up, from the north-east.
framed_config.world.sun_azimuth_deg = 15.0
framed_config.world.sun_elevation_deg = 30.0

sv.Scene().add_clouds(storm_clouds, microphysics="rams").to_blender(
    framed_bundle_dir, config=framed_config, render=True, quiet=True
)
setup.show_image(framed_bundle_dir)

## 4. Place the camera explicitly

Bounding-box framing fits everything exported. A storm with a long anvil
then pulls the camera far back. An explicit camera avoids that: set
`camera.location` and `camera.look_at` as (x, y, z) points in the model's
coordinates (m).

Here the camera stands 1 km above the ground, 30 km south of the storm's
core. It aims at a point 8 km up in the core. The core is the column with
the most condensate.

In [ ]:
# Total condensate path of each column (kg/kg summed over levels).
mixing_ratio_names = [name for name in storm_clouds.data_vars if name.startswith("R")]
condensate_path = sum(storm_clouds[name] for name in mixing_ratio_names).sum("z")

# The core: the column holding the most condensate.
core_column = condensate_path.where(
    condensate_path == condensate_path.max(), drop=True
)
core_x = float(core_column["x"][0])
core_y = float(core_column["y"][0])
print(f"storm core at x = {core_x / 1e3:.1f} km, y = {core_y / 1e3:.1f} km")

In [ ]:
explicit_bundle_dir = setup.WORK_DIR / "portrait_explicit"
explicit_config = sv.BlenderExportConfig.for_clouds(quality="preview")

# Camera 30 km south of the core, 1 km up, aimed 8 km up in the core.
camera_distance_south = 30.0e3
explicit_config.camera.location = (core_x, core_y - camera_distance_south, 1000.0)
explicit_config.camera.look_at = (core_x, core_y, 8000.0)
# A wide lens, so the whole tower fits.
explicit_config.camera.lens_mm = 24.0

explicit_config.world.sun_azimuth_deg = 15.0
explicit_config.world.sun_elevation_deg = 30.0

sv.Scene().add_clouds(storm_clouds, microphysics="rams").to_blender(
    explicit_bundle_dir, config=explicit_config, render=True, quiet=True
)
setup.show_image(explicit_bundle_dir)

## 5. Aerial perspective

Real air is not perfectly clear. Air molecules and aerosol scatter light
between the camera and a distant storm. Distant clouds therefore look paler
and bluer than near ones. `world.aerial_perspective = True` adds that haze
as a physically based volume. `world.visibility_km` sets how hazy it is:
the distance at which a dark object fades into the sky.

The haze makes a storm look far away. It also lowers the contrast of
everything in the image. Leave it off when detail matters more than
realism.

In [ ]:
haze_bundle_dir = setup.WORK_DIR / "portrait_haze"
haze_config = sv.BlenderExportConfig.for_clouds(quality="preview")

# The framing and the sun of section 3.
haze_config.camera.azimuth_deg = -55.0
haze_config.camera.elevation_deg = -8.0
haze_config.camera.distance_factor = 1.1
haze_config.world.sun_azimuth_deg = 15.0
haze_config.world.sun_elevation_deg = 30.0

# Haze with 60 km visibility.
haze_config.world.aerial_perspective = True
haze_config.world.visibility_km = 60.0

sv.Scene().add_clouds(storm_clouds, microphysics="rams").to_blender(
    haze_bundle_dir, config=haze_config, render=True, quiet=True
)
setup.show_image(haze_bundle_dir)

## 6. CM1 output with Morrison microphysics

CM1 writes one netCDF file per output time, numbered from 1 at t = 0.
`xarrams.read_cm1_output` reads them. CM1 needs two changes.

**Coordinates.** CM1 names its scalar-point coordinates `xh`, `yh` and
`zh`. Rename them to `x`, `y` and `z`. These runs store them in meters.
Some CM1 setups store kilometers, which must be converted.

**Species.** The Morrison scheme has settings that the output does not
record. `cm1_morrison_hydrometeor_species` takes them from the namelist:

- `droplet_number_concentration`: the fixed cloud droplet number, from
  `ndcnst` (cm^-3), converted to m^-3.
- `graupel_density`: 900 kg/m^3 for hail (`ihail = 1`), 400 for graupel.

The run below is an isolated storm in a 250 x 150 km domain. The crop keeps
the 100 x 90 km around the storm, so the export skips empty air.

In [ ]:
# Morrison mixing ratios (kg/kg), number concentrations (#/kg) and the
# air density (kg/m^3). Cloud droplet number is fixed, so there is no nc.
CM1_CLOUD_VARIABLES = ["qc", "qr", "qi", "qs", "qg", "nci", "ncr", "ncs", "ncg", "rho"]
# The crop around the storm (m), and the model time to render (s).
CM1_X_RANGE, CM1_Y_RANGE = (60.0e3, 160.0e3), (30.0e3, 120.0e3)
CM1_TIME_SECONDS = 2400.0


def cm1_output_file(run_dir, time_seconds):
    """
    The CM1 output file nearest a model time.

    CM1 numbers its files from 1 at t = 0, one per output interval. The
    interval is the time of the second file.
    """
    with xr.open_dataset(run_dir / "output" / "cm1out_000002.nc") as second_file:
        output_interval = float(second_file["time"].values[0])
    file_number = int(round(time_seconds / output_interval)) + 1
    return run_dir / "output" / f"cm1out_{file_number:06d}.nc"


# The output nearest 40 min: hydrometeors only, renamed, cropped to the storm.
cm1_output_path = cm1_output_file(setup.CM1_ISOLATED_STORM_RUN, CM1_TIME_SECONDS)
cm1_output = xarrams.read_cm1_output(
    [cm1_output_path], keep_vars=CM1_CLOUD_VARIABLES, silent=True
)
cm1_clouds = (
    cm1_output.isel(time=0)
    .rename({"xh": "x", "yh": "y", "zh": "z"})
    .sel(x=slice(*CM1_X_RANGE), y=slice(*CM1_Y_RANGE))
    .load()
)

# Morrison settings of this run's namelist: ndcnst = 300 cm^-3, ihail = 1.
cm1_species = cm1_morrison_hydrometeor_species(
    droplet_number_concentration=300.0e6, graupel_density=900.0
)
print(cm1_output_path.name, "variables:", ", ".join(cm1_clouds.data_vars))

In [ ]:
cm1_bundle_dir = setup.WORK_DIR / "portrait_cm1"
cm1_config = sv.BlenderExportConfig.for_clouds(quality="preview")

# Low and from the south-east, far enough back for the whole anvil.
cm1_config.camera.azimuth_deg = -60.0
cm1_config.camera.elevation_deg = -5.0
cm1_config.camera.distance_factor = 1.25
cm1_config.world.sun_azimuth_deg = -150.0
cm1_config.world.sun_elevation_deg = 30.0

# ``species`` replaces the ``microphysics`` preset.
sv.Scene().add_clouds(cm1_clouds, species=cm1_species).to_blender(
    cm1_bundle_dir, config=cm1_config, render=True, quiet=True
)
setup.show_image(cm1_bundle_dir)

## 7. From preview to final figure

The renders above are previews: 16 samples per pixel at 960 x 540. Cycles
removes most of the noise with its denoiser, but fine texture in thin cloud
suffers. For a final figure, use the "final" quality: 128 samples at
1920 x 1080. Either of these sets it:

- `sv.BlenderExportConfig.for_clouds()`, without `quality`;
- `config.render.set_quality("final")` on an existing config.

Set `config.render.resolution` afterwards for another image size. A
mature storm at the final quality takes about 10 minutes on 64 cores.

The bundle stays usable after the render. Open its `.blend` file in Blender
to adjust the camera or the light by hand. Rendering it again needs no
skyvista.

**Scripts with the same techniques:**

- `examples/rams_clouds/render_rams_cloud_still.py`
- `examples/cm1_storms/render_cm1_storm_portraits.py`. This script also
  shows a squall line lit from below its anvil, and a supercell isolated
  from its neighbours with tobac tracking.